In [1]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms.functional import resize
from torchvision.transforms import InterpolationMode
import random
from sklearn.model_selection import train_test_split

In [2]:
os.chdir('/home/ntdung/Medical')

In [6]:
excel_path = 'data/participants_1.xlsx'
df = pd.read_excel(excel_path, nrows=None)
df

,No.,subject_age,subject_dx,subject_sex,subject_id,dataset_name
0,1,44.2,control,m,sub-BrainAge000019,ABIDE/Caltech
1,2,39.3,control,m,sub-BrainAge000020,ABIDE/Caltech
2,3,42.5,control,m,sub-BrainAge000021,ABIDE/Caltech
3,4,19.7,control,m,sub-BrainAge000022,ABIDE/Caltech
4,5,20.0,control,f,sub-BrainAge000023,ABIDE/Caltech
...,...,...,...,...,...,...
4919,4920,66.0,control,f,sub-BrainAge023209,RocklandSample
4920,4921,69.0,control,m,sub-BrainAge023210,RocklandSample
4921,4922,23.0,control,m,sub-BrainAge023211,RocklandSample
4922,4923,54.0,control,f,sub-BrainAge023212,RocklandSample


In [8]:
def is_integer(n):
    return float(n).is_integer()

n_total = len(df)
n_integer = df['subject_age'].apply(is_integer).sum()
n_decimal = n_total - n_integer

print(f"Samples with integer age values: {n_integer}")
print(f"Samples with decimal age values: {n_decimal}")

Samples with integer age values: 4218
Samples with decimal age values: 706


In [9]:
df['subject_age'] = df['subject_age'].round().astype(int)
df

,No.,subject_age,subject_dx,subject_sex,subject_id,dataset_name
0,1,44,control,m,sub-BrainAge000019,ABIDE/Caltech
1,2,39,control,m,sub-BrainAge000020,ABIDE/Caltech
2,3,42,control,m,sub-BrainAge000021,ABIDE/Caltech
3,4,20,control,m,sub-BrainAge000022,ABIDE/Caltech
4,5,20,control,f,sub-BrainAge000023,ABIDE/Caltech
...,...,...,...,...,...,...
4919,4920,66,control,f,sub-BrainAge023209,RocklandSample
4920,4921,69,control,m,sub-BrainAge023210,RocklandSample
4921,4922,23,control,m,sub-BrainAge023211,RocklandSample
4922,4923,54,control,f,sub-BrainAge023212,RocklandSample


# Prepare Input

In [56]:
class BrainMRIDataset(Dataset):
    """
    Dataset cho Brain MRI với thông tin tuổi và giới tính thật và phản thực
    """
    def __init__(self, data_dir, participants_file, transform=None, img_size=128):
        self.data_dir = data_dir
        self.transform = transform
        self.img_size = img_size

        self.participants_df = pd.read_excel(participants_file)
        self.participants_df['subject_age'] = self.participants_df['subject_age'].round().astype(int)
        
        # Chuẩn hóa giới tính thành giá trị số
        self.participants_df['gender_code'] = self.participants_df['subject_sex'].map({'m': 0, 'f': 1})
        
        # Chuẩn hóa tuổi (min-max scaling)
        self.min_age = self.participants_df['subject_age'].min()
        self.max_age = self.participants_df['subject_age'].max()
        self.age_range = self.max_age - self.min_age
        
        # Lọc các mẫu có dữ liệu MRI hợp lệ
        valid_subjects = []
        for subject_id in self.participants_df['subject_id']:
            file_path = os.path.join(data_dir, subject_id, 'anat', f"{subject_id}_T1w.nii.gz")
            if os.path.exists(file_path):
                valid_subjects.append(subject_id)
        
        self.participants_df = self.participants_df[self.participants_df['subject_id'].isin(valid_subjects)]
        print(f"Tìm thấy {len(self.participants_df)} mẫu có dữ liệu MRI hợp lệ")
    
    def __len__(self):
        return len(self.participants_df)
    
    def _get_middle_slices(self, subject_id):
        try:
            file_path = os.path.join(self.data_dir, subject_id, 'anat', f"{subject_id}_T1w.nii.gz")
            img = nib.load(file_path)
            img_data = img.get_fdata()
            
            # Lấy 3 lát cắt giữa
            slices = [
                img_data[:, :, img_data.shape[2]//2],  # axial
                img_data[img_data.shape[0]//2, :, :],  # sagittal
                img_data[:, img_data.shape[1]//2, :]   # coronal
            ]
            
            ## Chuyển đổi sang tensor và chuẩn hóa
            processed_slices = []
            for slice_data in slices:
                tensor_slice = torch.from_numpy(slice_data).float()
                # Min-max normalization
                if tensor_slice.max() > tensor_slice.min():  # Tránh chia cho 0
                    tensor_slice = (tensor_slice - tensor_slice.min()) / (tensor_slice.max() - tensor_slice.min())
                    
                # Resize slice về kích thước cố định (128x128)
                tensor_slice = tensor_slice.unsqueeze(0)  # Thêm chiều kênh [1, H, W]
                resized_slice = F.interpolate(
                    tensor_slice.unsqueeze(0),  # Thêm chiều batch [1, 1, H, W]
                    size=(self.img_size, self.img_size),
                    mode='bilinear',
                    align_corners=False
                ).squeeze(0)  # Loại bỏ chiều batch [1, H, W]
                
                if self.transform:
                    resized_slice = self.transform(resized_slice)

                processed_slices.append(resized_slice)
            
            # Ghép các lát cắt thành một tensor [3, H, W]
            return torch.cat(processed_slices, dim=0)
            
        except Exception as e:
            print(f"Lỗi khi xử lý {subject_id}: {e}")
            # Trả về tensor 0 trong trường hợp lỗi
            return torch.zeros((3, 130, 130), dtype=torch.float32)
    
    def normalize_age(self, age):
        """Chuẩn hóa tuổi về khoảng [0, 1]"""
        return (age - self.min_age) / self.age_range if self.age_range > 0 else 0
    
    def __getitem__(self, idx):
        """
        Lấy một mẫu từ dataset
        """
        # Lấy thông tin mẫu thật
        real_info = self.participants_df.iloc[idx]
        real_id = real_info['subject_id']
        real_age = real_info['subject_age']
        real_gender = real_info['gender_code']
        
        # Lấy thông tin phản thực từ một mẫu ngẫu nhiên khác
        valid_indices = [i for i in range(len(self)) if i != idx]
        if not valid_indices:  # Nếu chỉ có 1 mẫu trong dataset
            cf_info = real_info
        else:
            cf_idx = random.choice(valid_indices)
            cf_info = self.participants_df.iloc[cf_idx]
        
        cf_id = cf_info['subject_id']
        cf_age = cf_info['subject_age']
        cf_gender = cf_info['gender_code']

        # Lấy ảnh và chuẩn hóa điều kiện
        real_img = self._get_middle_slices(real_id)
        
        # Chuẩn bị điều kiện
        real_condition = torch.tensor([self.normalize_age(real_age), real_gender], dtype=torch.float32)
        cf_condition = torch.tensor([self.normalize_age(cf_age), cf_gender], dtype=torch.float32)
        
        # Cũng chuẩn bị điều kiện gốc (không chuẩn hóa) cho việc kiểm tra/ghi log
        real_raw_condition = torch.tensor([real_age, real_gender], dtype=torch.float32)
        cf_raw_condition = torch.tensor([cf_age, cf_gender], dtype=torch.float32)
        
        return {
            'real_id': real_id,
            'cf_id': cf_id,
            'real_img': real_img,  # Shape: [3, H, W]
            'real_condition': real_condition,  # Shape: [2] - chuẩn hóa
            'cf_condition': cf_condition,  # Shape: [2] - chuẩn hóa
            'real_raw_condition': real_raw_condition,  # Shape: [2] - không chuẩn hóa
            'cf_raw_condition': cf_raw_condition  # Shape: [2] - không chuẩn hóa
        }

In [57]:
dataset = BrainMRIDataset(data_dir='data', participants_file='data/participants_1.xlsx')
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

Tìm thấy 4924 mẫu có dữ liệu MRI hợp lệ


In [58]:
sample = dataset[0]

print("Real ID:", sample['real_id'])
print("Counterfactual ID:", sample['cf_id'])
print("Real image shape:", sample['real_img'].shape)
print("Real condition (normalized):", sample['real_condition'])
print("Counterfactual condition (normalized):", sample['cf_condition'])
print("Real condition (raw):", sample['real_raw_condition'])
print("Counterfactual condition (raw):", sample['cf_raw_condition'])

Real ID: sub-BrainAge000019
Counterfactual ID: sub-BrainAge019714
Real image shape: torch.Size([3, 128, 128])
Real condition (normalized): tensor([0.3291, 0.0000])
Counterfactual condition (normalized): tensor([0.0253, 0.0000])
Real condition (raw): tensor([44.,  0.])
Counterfactual condition (raw): tensor([20.,  0.])


In [61]:
batch = next(iter(dataloader))

print("Real image batch shape:", batch['real_img'].shape)
print("Real condition (normalized):", batch['real_condition'], batch['real_condition'].shape)
print("Counterfactual condition (normalized):", batch['cf_condition'], batch['cf_condition'].shape)
print("Real IDs:", batch['real_id'])
print("Counterfactual IDs:", batch['cf_id'])

Real image batch shape: torch.Size([4, 3, 128, 128])
Real condition (normalized): tensor([[0.0633, 1.0000],
        [0.7215, 1.0000],
        [0.6709, 0.0000],
        [0.0253, 1.0000]]) torch.Size([4, 2])
Counterfactual condition (normalized): tensor([[0.0380, 1.0000],
        [0.6329, 1.0000],
        [0.0127, 1.0000],
        [0.6329, 0.0000]]) torch.Size([4, 2])
Real IDs: ['sub-BrainAge019054', 'sub-BrainAge022127', 'sub-BrainAge021273', 'sub-BrainAge019703']
Counterfactual IDs: ['sub-BrainAge011076', 'sub-BrainAge011088', 'sub-BrainAge022204', 'sub-BrainAge020652']


# Generator

In [62]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock(nn.Module):
    """Khối tích chập cơ bản cho U-Net"""
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x):
        return self.conv(x)

class SpatialTransformer(nn.Module):
    """Lớp biến đổi không gian để áp dụng biến dạng vào ảnh"""
    def __init__(self):
        super(SpatialTransformer, self).__init__()
    
    def forward(self, src, flow):
        """
        src: tensor hình ảnh nguồn [B, C, H, W]
        flow: tensor trường vector biến dạng [B, 2, H, W]
        """
        # Tạo lưới tọa độ chuẩn
        B, C, H, W = src.size()
        
        # Tạo lưới tọa độ chuẩn (từ -1 đến 1)
        grid_y, grid_x = torch.meshgrid(
            torch.linspace(-1, 1, H, device=src.device),
            torch.linspace(-1, 1, W, device=src.device),
            indexing='ij'  # Thêm indexing='ij' để tương thích với PyTorch mới
        )
        grid = torch.stack((grid_x, grid_y), dim=2).unsqueeze(0)
        grid = grid.expand(B, H, W, 2)
        
        # Chuyển đổi flow từ pixel sang giá trị chuẩn hóa (-1 đến 1)
        flow_grid = flow.permute(0, 2, 3, 1)  # [B, H, W, 2]
        flow_grid = flow_grid / torch.tensor([W/2, H/2], device=flow.device)
        
        # Cộng flow vào lưới cơ sở để tạo ra lưới biến dạng
        sample_grid = grid + flow_grid
        
        # Áp dụng lưới biến dạng để lấy mẫu từ ảnh gốc
        output = F.grid_sample(src, sample_grid, mode='bilinear', padding_mode='border', align_corners=True)
        
        return output

class ScalingAndSquaring(nn.Module):
    """Lớp scaling and squaring để tích hợp trường vận tốc thành trường biến dạng"""
    def __init__(self, n_steps=7):
        super(ScalingAndSquaring, self).__init__()
        self.n_steps = n_steps
        self.transformer = SpatialTransformer()
        
    def forward(self, velocity):
        """
        velocity: tensor trường vận tốc [B, 2, H, W]
        return: trường biến dạng phi(x) thông qua tích hợp scaling and squaring
        """
        # Chia nhỏ trường vận tốc
        flow = velocity / (2 ** self.n_steps)
        
        # Thực hiện scaling and squaring
        for _ in range(self.n_steps):
            flow = flow + self.transformer(flow, flow)
            
        return flow

class Generator(nn.Module):
    """Generator U-Net với scaling and squaring layers"""
    def __init__(self, in_channels=5, init_features=16):
        """
        in_channels: số kênh đầu vào (3 cho ảnh + 2 cho điều kiện)
        """
        super(Generator, self).__init__()
        
        # Encoder
        self.encoder1 = ConvBlock(in_channels, init_features)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.encoder2 = ConvBlock(init_features, init_features*2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.encoder3 = ConvBlock(init_features*2, init_features*2)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Bridge
        self.bridge = ConvBlock(init_features*2, init_features*2)
        
        # Decoder - Sử dụng Upsample thay vì ConvTranspose2d
        self.upconv3 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(init_features*2, init_features*2, kernel_size=3, padding=1)
        )
        self.decoder3 = ConvBlock(init_features*4, init_features*2)
        
        self.upconv2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(init_features*2, init_features*2, kernel_size=3, padding=1)
        )
        self.decoder2 = ConvBlock(init_features*4, init_features)
        
        self.upconv1 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(init_features, init_features, kernel_size=3, padding=1)
        )
        self.decoder1 = ConvBlock(init_features*2, init_features)
        
        # Velocity field prediction (2 kênh cho x và y)
        self.velocity = nn.Conv2d(init_features, 2, kernel_size=3, padding=1)
        
        # Scaling and squaring layer
        self.scaling_squaring = ScalingAndSquaring(n_steps=7)
        
        # Spatial transformer để áp dụng biến dạng
        self.transformer = SpatialTransformer()
        
    def forward(self, x, condition):
        """
        x: tensor ảnh đầu vào [B, 3, H, W]
        condition: tensor điều kiện mục tiêu [B, 2]
        """
        batch_size, _, H, W = x.size()
        
        # Mở rộng điều kiện thành kênh không gian
        condition = condition.view(batch_size, 2, 1, 1).expand(-1, -1, H, W)
        
        # Ghép ảnh và điều kiện
        x_in = torch.cat([x, condition], dim=1)  # [B, 5, H, W]

        # Lưu kích thước cho việc kiểm tra
        print(f"Input shape: {x_in.shape}")
        
        # Encoder
        enc1 = self.encoder1(x_in)
        enc1_pool = self.pool1(enc1)
        print(f"enc1 shape: {enc1.shape}, enc1_pool shape: {enc1_pool.shape}")
        
        enc2 = self.encoder2(enc1_pool)
        enc2_pool = self.pool2(enc2)
        print(f"enc2 shape: {enc2.shape}, enc2_pool shape: {enc2_pool.shape}")
        
        enc3 = self.encoder3(enc2_pool)
        enc3_pool = self.pool3(enc3)
        print(f"enc3 shape: {enc3.shape}, enc3_pool shape: {enc3_pool.shape}")
        
        # Bridge
        bridge = self.bridge(enc3_pool)
        print(f"bridge shape: {bridge.shape}")
        
        # Decoder với skip connections
        up3 = self.upconv3(bridge)
        print(f"up3 shape before concat: {up3.shape}")
        print(f"enc3 shape for concat: {enc3.shape}")
        
        # Đảm bảo kích thước khớp nhau
        if up3.shape[2:] != enc3.shape[2:]:
            up3 = F.interpolate(up3, size=enc3.shape[2:], mode='bilinear', align_corners=True)
            
        up3 = torch.cat([up3, enc3], dim=1)
        print(f"up3 shape after concat: {up3.shape}")
        dec3 = self.decoder3(up3)
        print(f"dec3 shape: {dec3.shape}")
        
        up2 = self.upconv2(dec3)
        print(f"up2 shape before concat: {up2.shape}")
        print(f"enc2 shape for concat: {enc2.shape}")
        
        # Đảm bảo kích thước khớp nhau
        if up2.shape[2:] != enc2.shape[2:]:
            up2 = F.interpolate(up2, size=enc2.shape[2:], mode='bilinear', align_corners=True)
            
        up2 = torch.cat([up2, enc2], dim=1)
        print(f"up2 shape after concat: {up2.shape}")
        dec2 = self.decoder2(up2)
        print(f"dec2 shape: {dec2.shape}")
        
        up1 = self.upconv1(dec2)
        print(f"up1 shape before concat: {up1.shape}")
        print(f"enc1 shape for concat: {enc1.shape}")
        
        # Đảm bảo kích thước khớp nhau
        if up1.shape[2:] != enc1.shape[2:]:
            up1 = F.interpolate(up1, size=enc1.shape[2:], mode='bilinear', align_corners=True)
            
        up1 = torch.cat([up1, enc1], dim=1)
        print(f"up1 shape after concat: {up1.shape}")
        dec1 = self.decoder1(up1)
        print(f"dec1 shape: {dec1.shape}")
        
        # Dự đoán trường vận tốc
        velocity = self.velocity(dec1)
        print(f"velocity shape: {velocity.shape}")
        
        # Tích hợp trường vận tốc để có trường biến dạng
        flow = self.scaling_squaring(velocity)
        print(f"flow shape: {flow.shape}")
        
        # Áp dụng trường biến dạng vào ảnh gốc
        deformed_image = self.transformer(x, flow)
        print(f"deformed_image shape: {deformed_image.shape}")
        
        return deformed_image, flow

class MultiSliceGenerator(nn.Module):
    """Mô hình Generator xử lý đồng thời 3 lát cắt MRI"""
    def __init__(self):
        super(MultiSliceGenerator, self).__init__()
        # Mỗi lát cắt được xử lý bởi một generator riêng
        self.generators = nn.ModuleList([
            Generator(in_channels=3+2)  # 3 channels cho ảnh + 2 cho điều kiện
            for _ in range(3)
        ])
        
    def forward(self, x, condition):
        """
        x: tensor ảnh gốc [B, 3, H, W]
        condition: tensor điều kiện mục tiêu [B, 2]
        """
        outputs = []
        flows = []
        
        # Xử lý mỗi lát cắt riêng biệt
        for i in range(3):
            slice_img = x[:, i:i+1, :, :]  # Lấy 1 lát cắt [B, 1, H, W]
            
            # Nhân rộng lát cắt thành 3 kênh để phù hợp với đầu vào của Generator
            slice_img = slice_img.repeat(1, 3, 1, 1)  # [B, 3, H, W]
            
            # Đưa vào generator
            print(f"\nProcessing slice {i+1}:")
            output, flow = self.generators[i](slice_img, condition)
            
            # Lấy kênh đầu tiên của output (vì cả 3 kênh giống nhau)
            output = output[:, 0:1, :, :]  # [B, 1, H, W]
            
            outputs.append(output)
            flows.append(flow)
        
        # Ghép 3 lát cắt đã xử lý
        generated_img = torch.cat(outputs, dim=1)  # [B, 3, H, W]
        
        return generated_img, flows

In [64]:
def test_generator():
    """Hàm kiểm tra Generator với kích thước đầu vào 128x128"""
    # Thiết lập seed cho tính khả lặp lại
    torch.manual_seed(42)
    
    # Tạo đầu vào giả
    batch_size = 2
    img_size = 128
    
    # Tạo ảnh đầu vào giả với 3 slice
    fake_img = torch.rand(batch_size, 3, img_size, img_size)
    
    # Tạo điều kiện giả (tuổi đã chuẩn hóa và giới tính)
    fake_condition = torch.tensor([[0.5, 0], [0.7, 1]], dtype=torch.float32)
    
    # Tạo mô hình
    model = MultiSliceGenerator()
    print("Mô hình MultiSliceGenerator đã được khởi tạo.")
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Discriminator total parameters: {total_params:,}")

    # Chuyển sang chế độ evaluation
    model.eval()
    
    # Thực hiện forward pass
    print("\nBắt đầu quá trình forward:")
    with torch.no_grad():
        output_img, output_flows = model(fake_img, fake_condition)
    
    # In kích thước đầu ra
    print("\nKết quả:")
    print(f"Input image shape: {fake_img.shape}")
    print(f"Output image shape: {output_img.shape}")
    print(f"Number of flow fields: {len(output_flows)}")
    for i, flow in enumerate(output_flows):
        print(f"Flow {i+1} shape: {flow.shape}")
    
    return output_img, output_flows

# Chạy hàm test
if __name__ == "__main__":
    test_generator()

Mô hình MultiSliceGenerator đã được khởi tạo.
Discriminator total parameters: 365,862

Bắt đầu quá trình forward:

Processing slice 1:
Input shape: torch.Size([2, 5, 128, 128])
enc1 shape: torch.Size([2, 16, 128, 128]), enc1_pool shape: torch.Size([2, 16, 64, 64])
enc2 shape: torch.Size([2, 32, 64, 64]), enc2_pool shape: torch.Size([2, 32, 32, 32])
enc3 shape: torch.Size([2, 32, 32, 32]), enc3_pool shape: torch.Size([2, 32, 16, 16])
bridge shape: torch.Size([2, 32, 16, 16])
up3 shape before concat: torch.Size([2, 32, 32, 32])
enc3 shape for concat: torch.Size([2, 32, 32, 32])
up3 shape after concat: torch.Size([2, 64, 32, 32])
dec3 shape: torch.Size([2, 32, 32, 32])
up2 shape before concat: torch.Size([2, 32, 64, 64])
enc2 shape for concat: torch.Size([2, 32, 64, 64])
up2 shape after concat: torch.Size([2, 64, 64, 64])
dec2 shape: torch.Size([2, 16, 64, 64])
up1 shape before concat: torch.Size([2, 16, 128, 128])
enc1 shape for concat: torch.Size([2, 16, 128, 128])
up1 shape after conca

# Discriminator

In [65]:
def weights_init_normal(model):
    '''
    Khởi tạo trọng số ổn định hơn cho mô hình so với khởi tạo mặc định của PyTorch
    :param model: mô hình cần khởi tạo
    :return:
    '''
    classname = model.__class__.__name__
    if classname.find("Conv") != -1:
        torch.nn.init.normal_(model.weight.data, 0.0, 0.02)

def conv_layer_2d(in_channel, out_channel, maxpool=True, kernel_size=3, padding=1, maxpool_stride=2):
    '''
    Tạo một block convolutional 2D gồm Conv2D, BatchNorm2D, (MaxPool2D tùy chọn), và ReLU
    '''
    if maxpool is True:
        layer = nn.Sequential(
            nn.Conv2d(in_channel, out_channel, padding=padding, kernel_size=kernel_size),
            nn.BatchNorm2d(out_channel),
            nn.MaxPool2d(2, stride=maxpool_stride),
            nn.ReLU(),
        )
    else:
        layer = nn.Sequential(
            nn.Conv2d(in_channel, out_channel, padding=padding, kernel_size=kernel_size),
            nn.BatchNorm2d(out_channel),
            nn.ReLU()
        )
    return layer

class Discriminator(nn.Module):

    def __init__(self, in_channels=3, channel_number=[16, 32, 64, 128, 256, 64]):
        '''
        Discriminator hoàn toàn convolutional cho GAN sinh ảnh phản thực
        :param in_channels: Số kênh đầu vào (3 lát cắt từ 3 trục)
        :param channel_number: Số kênh cho các lớp convolutional
        '''
        super(Discriminator, self).__init__()
        
        n_layer = len(channel_number)
        self.feature_extractor = nn.Sequential()
        
        # Xây dựng mạng trích xuất đặc trưng
        for i in range(n_layer):
            if i == 0:
                in_channel = in_channels  # Bắt đầu với số kênh đầu vào (3)
            else:
                in_channel = channel_number[i - 1]
            
            out_channel = channel_number[i]
            
            if i < n_layer - 1:
                # Sử dụng maxpool cho tất cả các lớp trừ lớp cuối
                self.feature_extractor.add_module(f'conv_{i}',
                                                  conv_layer_2d(in_channel,
                                                            out_channel,
                                                            maxpool=True,
                                                            kernel_size=3,
                                                            padding=1))
            else:
                # Lớp cuối không dùng maxpool
                self.feature_extractor.add_module(f'conv_{i}',
                                                  conv_layer_2d(in_channel,
                                                            out_channel,
                                                            maxpool=False,
                                                            kernel_size=1,
                                                            padding=0))

        in_channel = channel_number[-1]
        
        # Nhánh phân loại real/fake (adversarial)
        self.classifier_adv = nn.Sequential(
            nn.Conv2d(in_channel, 1, kernel_size=3, padding=0, bias=False),
            nn.AdaptiveAvgPool2d(1),  # Global average pooling để giảm kích thước xuống 1x1
            nn.Flatten(),
            nn.Sigmoid()  # Sigmoid cho phân loại nhị phân (real/fake)
        )
        
        # Nhánh phân loại giới tính
        self.classifier_gender = nn.Sequential(
            nn.Conv2d(in_channel, 8, kernel_size=3, padding=0, bias=False),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),  # Global average pooling
            nn.Flatten(),
            nn.Linear(8, 1),
            nn.Sigmoid()  # Sigmoid cho phân loại nhị phân (nam/nữ)
        )
        
        # Nhánh hồi quy tuổi
        self.classifier_age = nn.Sequential(
            nn.Conv2d(in_channel, 16, kernel_size=3, padding=0, bias=False),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),  # Global average pooling
            nn.Flatten(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)  # Không có activation để dự đoán giá trị thực
        )

    def forward(self, x):
        '''
        :param x: Tensor hình ảnh đầu vào có dạng (batch_size, 3, H, W)
        :return: Tuple gồm (adv_preds, gender_preds, age_preds) tương ứng với 3 đầu ra
        '''
        # Trích xuất đặc trưng chung cho cả 3 tác vụ
        encoded_features = self.feature_extractor(x)
        
        # Áp dụng 3 nhánh dự đoán
        adv_preds = self.classifier_adv(encoded_features)
        gender_preds = self.classifier_gender(encoded_features)
        age_preds = self.classifier_age(encoded_features)
        
        return adv_preds, gender_preds, age_preds

In [66]:
# Hàm kiểm tra kích thước đầu ra
def test_discriminator():
    batch_size = 4
    height, width = 128, 128  # Giả sử kích thước ảnh đầu vào là 128x128
    
    # Tạo tensor đầu vào ngẫu nhiên
    x = torch.randn(batch_size, 3, height, width)
    
    # Khởi tạo discriminator
    discriminator = Discriminator()
    
    total_params = sum(p.numel() for p in discriminator.parameters())
    print(f"Discriminator total parameters: {total_params:,}")
          
    # Áp dụng discriminator
    adv_preds, gender_preds, age_preds = discriminator(x)
    
    # In kích thước đầu ra
    print(f"Discriminator input shape: {x.shape}")
    print(f"Adversarial predictions shape: {adv_preds.shape}")  # Kỳ vọng: [batch_size, 1]
    print(f"Gender predictions shape: {gender_preds.shape}")    # Kỳ vọng: [batch_size, 1]
    print(f"Age predictions shape: {age_preds.shape}")          # Kỳ vọng: [batch_size, 1]
    
    return discriminator

if __name__ == "__main__":
    test_discriminator()

Discriminator total parameters: 424,730
Discriminator input shape: torch.Size([4, 3, 128, 128])
Adversarial predictions shape: torch.Size([4, 1])
Gender predictions shape: torch.Size([4, 1])
Age predictions shape: torch.Size([4, 1])


# Training

In [ ]:
import torch.optim as optim

class BrainGAN(nn.Module):
    """Mô hình GAN hoàn chỉnh kết hợp Generator và Discriminator"""
    def __init__(self, lr_g=0.0002, lr_d=0.0002, betas=(0.5, 0.999)):
        super(BrainGAN, self).__init__()
        
        # Khởi tạo generator và discriminator
        self.generator = MultiSliceGenerator()
        self.discriminator = Discriminator(in_channels=3)
        
        # Áp dụng hàm khởi tạo trọng số
        self.generator.apply(weights_init_normal)
        self.discriminator.apply(weights_init_normal)
        
        # Khởi tạo optimizers
        self.optimizer_G = optim.Adam(self.generator.parameters(), lr=lr_g, betas=betas)
        self.optimizer_D = optim.Adam(self.discriminator.parameters(), lr=lr_d, betas=betas)
        
        # Định nghĩa các hàm mất mát
        self.adversarial_loss = nn.BCELoss()
        self.gender_loss = nn.BCELoss()
        self.age_loss = nn.MSELoss()
        self.pixel_loss = nn.L1Loss()  # L1 loss để giữ chi tiết ảnh
        
    def train_discriminator(self, real_imgs, gender_labels, age_labels):
        """
        Huấn luyện discriminator
        :param real_imgs: Ảnh thật [B, 3, H, W]
        :param gender_labels: Nhãn giới tính [B, 1]
        :param age_labels: Nhãn tuổi [B, 1]
        """
        batch_size = real_imgs.size(0)
        
        # Tạo nhãn thật/giả
        valid = torch.ones(batch_size, 1, device=real_imgs.device)
        fake = torch.zeros(batch_size, 1, device=real_imgs.device)
        
        # Tạo điều kiện ngẫu nhiên (giới tính và tuổi)
        random_conditions = torch.cat([
            torch.rand(batch_size, 1, device=real_imgs.device),  # Giới tính ngẫu nhiên
            torch.rand(batch_size, 1, device=real_imgs.device) * 80  # Tuổi 0-80
        ], dim=1)
        
        # Tạo ảnh giả từ generator
        fake_imgs, _ = self.generator(real_imgs, random_conditions)
        
        # ----- Huấn luyện với ảnh thật -----
        self.optimizer_D.zero_grad()
        
        # Dự đoán từ discriminator với ảnh thật
        real_adv_preds, real_gender_preds, real_age_preds = self.discriminator(real_imgs)
        
        # Tính các thành phần mất mát trên ảnh thật
        d_real_adv_loss = self.adversarial_loss(real_adv_preds, valid)
        d_real_gender_loss = self.gender_loss(real_gender_preds, gender_labels)
        d_real_age_loss = self.age_loss(real_age_preds, age_labels)
        
        # Tổng mất mát trên ảnh thật
        d_real_loss = d_real_adv_loss + d_real_gender_loss + d_real_age_loss
        
        # ----- Huấn luyện với ảnh giả -----
        # Dự đoán từ discriminator với ảnh giả
        fake_adv_preds, _, _ = self.discriminator(fake_imgs.detach())
        
        # Tính mất mát trên ảnh giả
        d_fake_loss = self.adversarial_loss(fake_adv_preds, fake)
        
        # Tổng mất mát discriminator
        d_loss = d_real_loss + d_fake_loss
        
        # Lan truyền ngược và cập nhật
        d_loss.backward()
        self.optimizer_D.step()
        
        return {
            "d_loss": d_loss.item(),
            "d_real_adv_loss": d_real_adv_loss.item(),
            "d_fake_loss": d_fake_loss.item(),
            "d_real_gender_loss": d_real_gender_loss.item(),
            "d_real_age_loss": d_real_age_loss.item()
        }
        
    def train_generator(self, real_imgs, gender_labels, age_labels):
        """
        Huấn luyện generator
        :param real_imgs: Ảnh thật [B, 3, H, W]
        :param gender_labels: Nhãn giới tính [B, 1]
        :param age_labels: Nhãn tuổi [B, 1]
        """
        batch_size = real_imgs.size(0)
        
        # Tạo nhãn thật
        valid = torch.ones(batch_size, 1, device=real_imgs.device)
        
        # Sử dụng nhãn thật làm điều kiện
        conditions = torch.cat([gender_labels, age_labels], dim=1)
        
        self.optimizer_G.zero_grad()
        
        # Tạo ảnh giả
        fake_imgs, flows = self.generator(real_imgs, conditions)
        
        # Dự đoán từ discriminator với ảnh giả
        fake_adv_preds, fake_gender_preds, fake_age_preds = self.discriminator(fake_imgs)
        
        # Tính các thành phần mất mát
        g_adv_loss = self.adversarial_loss(fake_adv_preds, valid)  # Mất mát đối kháng
        g_gender_loss = self.gender_loss(fake_gender_preds, gender_labels)  # Mất mát giới tính
        g_age_loss = self.age_loss(fake_age_preds, age_labels)  # Mất mát tuổi
        
        # Mất mát định danh (identity loss) - Tương tự điều kiện khi điều kiện giống với ảnh nguồn
        # Lấy giới tính và tuổi từ ảnh thật
        real_adv_preds, real_gender_preds, real_age_preds = self.discriminator(real_imgs)
        identity_conditions = torch.cat([real_gender_preds.detach(), real_age_preds.detach()], dim=1)
        identity_imgs, _ = self.generator(real_imgs, identity_conditions)
        identity_loss = self.pixel_loss(identity_imgs, real_imgs)
        
        # Tổng mất mát generator
        g_loss = g_adv_loss + g_gender_loss + g_age_loss + 10 * identity_loss
        
        # Lan truyền ngược và cập nhật
        g_loss.backward()
        self.optimizer_G.step()
        
        return {
            "g_loss": g_loss.item(),
            "g_adv_loss": g_adv_loss.item(),
            "g_gender_loss": g_gender_loss.item(),
            "g_age_loss": g_age_loss.item(),
            "identity_loss": identity_loss.item()
        }
    
    def generate(self, input_imgs, conditions):
        """
        Phương thức tạo ảnh mới từ generator
        :param input_imgs: Ảnh đầu vào [B, 3, H, W]
        :param conditions: Điều kiện [B, 2] (giới tính, tuổi)
        """
        self.generator.eval()
        with torch.no_grad():
            fake_imgs, flows = self.generator(input_imgs, conditions)
        return fake_imgs, flows
    
    def predict(self, imgs):
        """
        Dự đoán thuộc tính từ discriminator
        :param imgs: Ảnh đầu vào [B, 3, H, W]
        """
        self.discriminator.eval()
        with torch.no_grad():
            adv_preds, gender_preds, age_preds = self.discriminator(imgs)
        return adv_preds, gender_preds, age_preds


# ------------ TRAINING LOOP EXAMPLE ------------

def train_brain_gan(model, train_loader, num_epochs=100, device='cuda'):
    """
    Vòng lặp huấn luyện cho BrainGAN
    :param model: Mô hình BrainGAN
    :param train_loader: DataLoader chứa dữ liệu huấn luyện
    :param num_epochs: Số epoch
    :param device: Thiết bị ('cuda' hoặc 'cpu')
    """
    model = model.to(device)
    
    for epoch in range(num_epochs):
        d_losses = []
        g_losses = []
        
        for i, (imgs, gender_labels, age_labels) in enumerate(train_loader):
            # Chuyển dữ liệu tới thiết bị
            imgs = imgs.to(device)
            gender_labels = gender_labels.to(device)
            age_labels = age_labels.to(device)
            
            # Huấn luyện discriminator
            d_loss_dict = model.train_discriminator(imgs, gender_labels, age_labels)
            d_losses.append(d_loss_dict["d_loss"])
            
            # Huấn luyện generator
            g_loss_dict = model.train_generator(imgs, gender_labels, age_labels)
            g_losses.append(g_loss_dict["g_loss"])
            
            # In thông tin huấn luyện
            if i % 50 == 0:
                print(f"[Epoch {epoch}/{num_epochs}] [Batch {i}/{len(train_loader)}] "
                      f"[D loss: {d_loss_dict['d_loss']:.4f}] [G loss: {g_loss_dict['g_loss']:.4f}]")
                
        avg_d_loss = sum(d_losses) / len(d_losses)
        avg_g_loss = sum(g_losses) / len(g_losses)
        
        print(f"[Epoch {epoch}/{num_epochs}] [D loss: {avg_d_loss:.4f}] [G loss: {avg_g_loss:.4f}]")
        
        # Lưu mô hình
        if epoch % 10 == 0:
            torch.save(model.state_dict(), f"brain_gan_epoch_{epoch}.pth")


# ------------ EXAMPLE USAGE ------------

def print_model_architecture(model):
    """In thông tin kiến trúc mô hình"""
    # Tính tổng số tham số
    generator_params = sum(p.numel() for p in model.generator.parameters())
    discriminator_params = sum(p.numel() for p in model.discriminator.parameters())
    total_params = generator_params + discriminator_params
    
    print("\n" + "="*50)
    print("KIẾN TRÚC BRAIN GAN")
    print("="*50)
    
    print("\nTHÔNG TIN TỔNG QUAN:")
    print(f"Tổng số tham số: {total_params:,}")
    print(f"Generator: {generator_params:,} tham số")
    print(f"Discriminator: {discriminator_params:,} tham số")

In [55]:
def main():
    # Khởi tạo mô hình
    model = BrainGAN(lr_g=0.0002, lr_d=0.0002)
    
    # Giả sử chúng ta có DataLoader hợp lệ
    # train_loader = get_brain_mri_dataloader(...)
    
    # Huấn luyện mô hình
    # train_brain_gan(model, train_loader, num_epochs=100, device='cuda')
    
    # In thông tin kiến trúc
    print_model_architecture(model)

if __name__ == "__main__":
    main()

AttributeError: 'ConvBlock' object has no attribute 'weight'

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset
from torchinfo import summary


# Hàm tạo mô hình GAN tổng thể
class BrainAgeGAN(nn.Module):
    def __init__(self):
        super(BrainAgeGAN, self).__init__()
        # Khởi tạo generator và discriminator
        self.generator = MultiSliceGenerator()
        self.discriminator = BrainGANDiscriminator()
        
        # Áp dụng hàm khởi tạo trọng số
        self.generator.apply(weights_init_normal)
        self.discriminator.apply(weights_init_normal)
    
    def forward(self, x, condition):
        """
        x: tensor ảnh gốc [B, 3, H, W]
        condition: tensor điều kiện mục tiêu [B, 2]
        """
        # Generator tạo ảnh giả
        generated_images, flows = self.generator(x, condition)
        
        # Discriminator phân loại ảnh thật và ảnh giả
        fake_adv, fake_gender, fake_age = self.discriminator(generated_images)
        real_adv, real_gender, real_age = self.discriminator(x)
        
        return {
            'generated_images': generated_images,
            'flows': flows,
            'fake_adv': fake_adv,
            'fake_gender': fake_gender,
            'fake_age': fake_age,
            'real_adv': real_adv, 
            'real_gender': real_gender,
            'real_age': real_age
        }

# Hàm kiểm tra tính tương thích giữa Generator và Discriminator
def test_gan_compatibility():
    # Tạo dữ liệu mẫu
    batch_size = 4
    height, width = 128, 128
    
    # Input tensor là hình ảnh MRI với 3 lát cắt
    input_images = torch.randn(batch_size, 3, height, width)
    
    # Điều kiện target là mảng 2 chiều: [tuổi, giới tính]
    # Tuổi được chuẩn hóa về [0,1], giới tính là 0 hoặc 1
    target_conditions = torch.tensor([
        [0.5, 1],  # tuổi 50%, giới tính nam
        [0.7, 0],  # tuổi 70%, giới tính nữ
        [0.3, 1],  # tuổi 30%, giới tính nam
        [0.6, 0],  # tuổi 60%, giới tính nữ
    ], dtype=torch.float32)
    
    # Khởi tạo generator
    generator = MultiSliceGenerator()
    print("\n===== GENERATOR TEST =====")
    
    # Tắt tạm thời các print debug trong Generator
    import sys
    original_stdout = sys.stdout
    sys.stdout = open('generator_debug.log', 'w')
    
    try:
        # Sinh ảnh giả từ generator
        generated_images, flows = generator(input_images, target_conditions)
        
        # Khôi phục stdout
        sys.stdout = original_stdout
        
        print(f"Generator input shape: {input_images.shape}")
        print(f"Target conditions shape: {target_conditions.shape}")
        print(f"Generated images shape: {generated_images.shape}")
        print(f"Number of flows: {len(flows)}")
        print(f"Flow shape: {flows[0].shape}")
        
        # Khởi tạo discriminator
        discriminator = BrainGANDiscriminator()
        print("\n===== DISCRIMINATOR TEST =====")
        
        # Áp dụng discriminator lên ảnh thật
        real_adv, real_gender, real_age = discriminator(input_images)
        print(f"Real adversarial output shape: {real_adv.shape}")
        print(f"Real gender output shape: {real_gender.shape}")
        print(f"Real age output shape: {real_age.shape}")
        
        # Áp dụng discriminator lên ảnh giả
        fake_adv, fake_gender, fake_age = discriminator(generated_images)
        print(f"Fake adversarial output shape: {fake_adv.shape}")
        print(f"Fake gender output shape: {fake_gender.shape}")
        print(f"Fake age output shape: {fake_age.shape}")
        
        print("\n✓ Generator và Discriminator tương thích với nhau!")
        
        # Đếm số tham số của từng mô hình
        gen_params = sum(p.numel() for p in generator.parameters())
        disc_params = sum(p.numel() for p in discriminator.parameters())
        total_params = gen_params + disc_params
        
        print(f"\nTổng số tham số Generator: {gen_params:,}")
        print(f"Tổng số tham số Discriminator: {disc_params:,}")
        print(f"Tổng số tham số GAN: {total_params:,}")
        
        return generator, discriminator
        
    except Exception as e:
        # Khôi phục stdout trong trường hợp lỗi
        sys.stdout = original_stdout
        print(f"Lỗi khi kiểm tra tính tương thích: {e}")
        raise e

# Hàm tổng hợp kiến trúc mô hình
def print_model_summary():
    try:
        # Sử dụng torchinfo để hiển thị thông tin chi tiết hơn so với torch.summary
        batch_size = 4
        height, width = 128, 128
        
        # Tạo input
        input_images = torch.randn(batch_size, 3, height, width)
        target_conditions = torch.randn(batch_size, 2)
        
        print("\n===== GENERATOR SUMMARY =====")
        generator = MultiSliceGenerator()
        summary(generator, input_data=[input_images, target_conditions])
        
        print("\n===== DISCRIMINATOR SUMMARY =====")
        discriminator = BrainGANDiscriminator()
        summary(discriminator, input_data=torch.randn(batch_size, 3, height, width))
        
        print("\n===== COMPLETE GAN SUMMARY =====")
        gan_model = BrainAgeGAN()
        summary(gan_model, input_data=[input_images, target_conditions])
        
    except Exception as e:
        print(f"Lỗi khi tạo tổng hợp mô hình: {e}")

In [43]:
test_gan_compatibility()
print_model_summary()


===== GENERATOR TEST =====
Generator input shape: torch.Size([4, 3, 128, 128])
Target conditions shape: torch.Size([4, 2])
Generated images shape: torch.Size([4, 3, 128, 128])
Number of flows: 3
Flow shape: torch.Size([4, 2, 128, 128])

===== DISCRIMINATOR TEST =====
Real adversarial output shape: torch.Size([4, 1])
Real gender output shape: torch.Size([4, 1])
Real age output shape: torch.Size([4, 1])
Fake adversarial output shape: torch.Size([4, 1])
Fake gender output shape: torch.Size([4, 1])
Fake age output shape: torch.Size([4, 1])

✓ Generator và Discriminator tương thích với nhau!

Tổng số tham số Generator: 365,862
Tổng số tham số Discriminator: 424,730
Tổng số tham số GAN: 790,592

===== GENERATOR SUMMARY =====

Processing slice 1:
Input shape: torch.Size([4, 5, 128, 128])
enc1 shape: torch.Size([4, 16, 128, 128]), enc1_pool shape: torch.Size([4, 16, 64, 64])
enc2 shape: torch.Size([4, 32, 64, 64]), enc2_pool shape: torch.Size([4, 32, 32, 32])
enc3 shape: torch.Size([4, 32, 32

In [13]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.distributions.normal import Normal
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import random

# Thiết lập seed để kết quả tái tạo được
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

os.chdir('/home/ntdung/Medical')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
def generate_brain_images(generator, age, gender, latent_dim=100, num_samples=5):
    """
    Tạo ảnh não dựa trên tuổi và giới tính
    
    Args:
        generator: Generator đã huấn luyện
        age: Tuổi (số thực)
        gender: Giới tính ('m' hoặc 'f')
        latent_dim: Kích thước latent vector
        num_samples: Số lượng mẫu cần tạo
    """
    generator.eval()
    with torch.no_grad():
        # Tạo nhiễu ngẫu nhiên
        z = torch.randn(num_samples, latent_dim, device=device)
        
        # Chuẩn bị điều kiện
        norm_age = age / 100.0  # Chuẩn hóa tuổi
        gender_val = 1.0 if gender == 'm' else 0.0
        condition = torch.tensor([[norm_age, gender_val]]).float().to(device)
        condition = condition.repeat(num_samples, 1)
        
        # Tạo ảnh
        generated_images = generator(z, condition)
        
        # Hiển thị kết quả
        fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))
        
        for i in range(num_samples):
            for j, slice_name in enumerate(['Axial', 'Coronal', 'Sagittal']):
                if num_samples > 1:
                    ax = axes[i, j]
                else:
                    ax = axes[j]
                
                # Chuyển từ [-1, 1] về [0, 1] để hiển thị
                img = (generated_images[i, j].cpu().numpy() + 1) / 2
                ax.imshow(img, cmap='gray')
                ax.set_title(f"{slice_name} Slice")
                ax.axis('off')
        
        gender_str = "Nam" if gender == 'm' else "Nữ"
        plt.suptitle(f'Ảnh MRI não được tạo: Tuổi {age}, Giới tính {gender_str}')
        plt.tight_layout()
        plt.show()

# Thử nghiệm tạo ảnh
try:
    # Tải model đã huấn luyện (nếu có)
    checkpoint_path = 'model_checkpoint_epoch_99.pth'  # Đổi tên file nếu cần
    
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path)
        generator.load_state_dict(checkpoint['generator'])
        print("Đã tải model đã huấn luyện!")
    
    # Tạo một số ảnh với các tham số khác nhau
    print("Tạo ảnh cho nam 35 tuổi:")
    generate_brain_images(generator, age=35, gender='m', num_samples=3)
    
    print("Tạo ảnh cho nữ 70 tuổi:")
    generate_brain_images(generator, age=70, gender='f', num_samples=3)
    
except Exception as e:
    print(f"Lỗi khi tạo ảnh: {e}")
    print("Bạn cần huấn luyện mô hình trước khi tạo ảnh.")

In [ ]:
def save_model(generator, discriminator, optimizer_G, optimizer_D, epoch, filename='gan_model.pth'):
    """Lưu trạng thái mô hình GAN"""
    torch.save({
        'generator': generator.state_dict(),
        'discriminator': discriminator.state_dict(),
        'optimizer_G': optimizer_G.state_dict(),
        'optimizer_D': optimizer_D.state_dict(),
        'epoch': epoch
    }, filename)
    print(f"Đã lưu mô hình vào {filename}")

def load_model(generator, discriminator, optimizer_G=None, optimizer_D=None, filename='gan_model.pth'):
    """Tải trạng thái mô hình GAN"""
    if os.path.exists(filename):
        checkpoint = torch.load(filename)
        generator.load_state_dict(checkpoint['generator'])
        discriminator.load_state_dict(checkpoint['discriminator'])
        
        if optimizer_G is not None:
            optimizer_G.load_state_dict(checkpoint['optimizer_G'])
        if optimizer_D is not None:
            optimizer_D.load_state_dict(checkpoint['optimizer_D'])
            
        epoch = checkpoint['epoch']
        print(f"Đã tải mô hình từ epoch {epoch}")
        return epoch
    else:
        print(f"Không tìm thấy file {filename}")
        return 0

# Ví dụ cách sử dụng
try:
    # Khởi tạo optimizer mới (chỉ để minh họa)
    optimizer_G = optim.Adam(generator.parameters(), lr=lr_g, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=lr_d, betas=(0.5, 0.999))
    
    # Lưu mô hình
    save_model(generator, discriminator, optimizer_G, optimizer_D, num_epochs, 'gan_model_final.pth')
    
    # Tải mô hình (chỉ để minh họa)
    start_epoch = load_model(generator, discriminator, optimizer_G, optimizer_D, 'gan_model_final.pth')
    print(f"Tiếp tục huấn luyện từ epoch {start_epoch}")
    
except Exception as e:
    print(f"Lỗi khi lưu/tải mô hình: {e}")

print("Hoàn thành code cho dự án GAN sinh ảnh não dựa trên điều kiện tuổi và giới tính!")